In NLP latest architecture is Transformer based architectures There's three types. Encoder-only models like BERT, decoder-only models like GPT and llama. There's encoder-decoder architectures as well: Encoder-decoder models can be implemented using various neural network architectures, and their names often reflect the type of network used or the specific application. Here are some key examples:
1. Transformer-based Encoder-Decoder Models:
These models are based on the Transformer architecture, which has significantly advanced the field of natural language processing.
T5 (Text-to-Text Transfer Transformer): T5 is a Transformer-based encoder-decoder model that treats all NLP tasks as text-to-text problems. It's widely used for tasks like machine translation, summarization, and question answering.
BART (Bidirectional and Auto-Regressive Transformer): BART is another Transformer-based encoder-decoder model that excels at denoising sequence-to-sequence tasks. It's particularly effective for text generation and summarization.
Pegasus (Pre-training with Extracted Gap-sentences for Abstractive Summarization): Pegasus is a Transformer-based encoder-decoder model specifically designed for abstractive summarization, where it generates summaries by focusing on key sentences extracted from the input text.
MT5 (Massively Multilingual Text-to-Text Transformer): MT5 is a multilingual variant of T5, trained on a large corpus of text in various languages.
FLAN-T5 (Scaling Instruction-Finetuned Language Models): FLAN-T5 is an extension of T5 that has been finetuned on a wide range of tasks and instructions to improve its generalization capabilities.
Code-T5: This is a variant of T5 designed specifically for code understanding and generation.
UL2 (Unifying Language Learning Paradigms): UL2 is another Transformer-based encoder-decoder model with a unified approach to language learning.
FLAN-UL2: This is a finetuned version of UL2 with improved performance on various tasks.
EdgeFormer: A Transformer-based encoder-decoder model designed for efficient seq2seq generation on devices with limited resources.
2. Models Utilizing Encoder-Decoder Architecture for Specific Tasks:
Encoder-decoder architectures can also be used as components within larger models designed for specific tasks:
VisionEncoderDecoderModel: This model initializes an image-to-text model with a pretrained vision model (like ViT) as the encoder and a pretrained language model (like BERT or GPT2) as the decoder. This allows it to perform tasks like image captioning and optical character recognition (OCR).
TrOCR (Transformer-based Optical Character Recognition): TrOCR is a specific instance of the VisionEncoderDecoderModel architecture, fine-tuned for OCR.
Note: Encoder-decoder architecture is a framework, and specific implementations can vary in their internal network structure (RNN, CNN, Transformer) and pre-training objectives.

LangChain can be used for app development. Web-side of app development is still done with FastAPI. If you want to include agents then Langgraph is used. Fine-tuning of the model for best hyperparameters can be done using LoRA or QLoRA. LoRA is where memory is a constraint but want to maintain high precision. QLoRA (quantized lora) is used to optimize memory efficiency in comprise for a minimal loss in performance.

Llama 2 and 3 and mistral ai were attempted but did not work with system ram and gpu usage constraints. Tiny Llama was finally leveraged on a dataset that contained two columns: one with a full clinical note and the other with the summary written by a healthcare provider. The llm was given the task to summarize a given synthetic clinical note by chatgpt4. The result was an accurate response with extra unnecessary details

In [1]:
!pip install triton==2.1.0 bitsandbytes==0.41.0 peft==0.7.0 transformers==4.38.2 accelerate==0.30.0 trl==0.4.7

ERROR: Could not find a version that satisfies the requirement triton==2.1.0 (from versions: none)
ERROR: No matching distribution found for triton==2.1.0


In [2]:
!pip install huggingface_hub
!pip install numpy
import numpy as np

In [3]:
!pip install peft

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 680.7/680.7 kB 1.4 MB/s  0:00:0036m-:--:--


In [4]:
#!pip install triton
!pip install trl
import torch
from transformers import (AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments, pipeline)
from transformers.generation import LogitsProcessorList, TopKLogitsWarper, TopPLogitsWarper
from trl import SFTTrainer
from peft import LoraConfig
from datasets import load_dataset

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 697.4/697.4 kB 12.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 28.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13/13 [trl]32m12/13 [trl]sets]


Llama model requires too much system ram and is crashing which is the reason for using tiny llama.

In [16]:
# Install (keep versions compatible)
!pip install -U transformers datasets peft accelerate trl

import torch
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    DataCollatorForLanguageModeling,
    Trainer
)
from peft import LoraConfig, get_peft_model

# ---------------------------
# 1. Load model + tokenizer
# ---------------------------
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(model_name)

model.config.pad_token_id = tokenizer.pad_token_id
model.config.use_cache = False

# ---------------------------
# 2. Apply LoRA (CRITICAL)
# ---------------------------
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)

# ---------------------------
# 3. Load + format dataset
# ---------------------------
dataset = load_dataset("geekdom/clinical_data", split="train")

# Use smaller subset for faster training (optional)
dataset = dataset.select(range(3000))

def format_example(example):
    return {
        "text": f"Summarize this clinical note:\n{example['prompt']}\n\nSummary:\n{example['response']}"
    }

dataset = dataset.map(format_example)
dataset = dataset.remove_columns([col for col in dataset.column_names if col != "text"])

# ---------------------------
# 4. Tokenization
# ---------------------------
def tokenize(example):
    return tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=256   # smaller = faster
    )

dataset = dataset.map(tokenize, batched=True)

def add_labels(example):
    example["labels"] = example["input_ids"].copy()
    return example

dataset = dataset.map(add_labels, batched=True)
dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

# ---------------------------
# 5. Training setup
# ---------------------------
training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,   # simulate batch size 4
    max_steps=200,                  # MUCH better than 4
    logging_steps=10,
    save_steps=50,
    learning_rate=2e-4,
    dataloader_pin_memory=False
)

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    tokenizer=tokenizer,
    data_collator=data_collator
)

# ---------------------------
# 6. Train
# ---------------------------
trainer.train()

# ---------------------------
# 7. Inference (FIXED)
# ---------------------------
device = "mps" if torch.backends.mps.is_available() else "cpu"
model.to(device)

prompt = """
Summarize the following clinical note:

Hospital Course:
The patient was admitted to the ICU six days after testing positive for COVID-19 due to worsening respiratory distress.
Early physical therapy and prone positioning improved oxygen saturation from 85% to 94%.
After five days, the patient was transferred to the general ward.

Discharge Condition:
The patient was stable, breathing comfortably, and able to walk short distances.

Summary:
"""

inputs = tokenizer(prompt, return_tensors="pt").to(device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=120,
        do_sample=True,        # IMPORTANT
        temperature=0.7,
        top_p=0.9,
        pad_token_id=tokenizer.pad_token_id
    )

# ONLY print generated part (FIX)
generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]
response = tokenizer.decode(generated_tokens, skip_special_tokens=True)

print("\nGenerated Summary:\n")
print(response)

  Using cached huggingface_hub-1.11.0-py3-none-any.whl.metadata (14 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 36.3 MB/s  0:00:00eta 0:00:01
Using cached huggingface_hub-1.11.0-py3-none-any.whl (645 kB)
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 0.36.2
    Uninstalling huggingface_hub-0.36.2:
      Successfully uninstalled huggingface_hub-0.36.2
  Attempting uninstall: accelerate━━━━━━━━━━━━━━ 0/3 [huggingface-hub]
    Found existing installation: accelerate 1.12.00/3 [huggingface-hub]
    Uninstalling accelerate-1.12.0:━━━━━━━━━ 0/3 [huggingface-hub]
      Successfully uninstalled accelerate-1.12.0 0/3 [huggingface-hub]
  Attempting uninstall: transformers━━━━━━━━ 0/3 [huggingface-hub]
    Found existing installation: transformers 4.57.62m0/3 [huggingface-hub]
    Uninstalling transformers-4.57.6:m╸━━━━━━━━━━━━━ 2/3 [transformers]
      Successfully uninstalled transformers-4.57.6━━━━━━━━━━━━ 2/3 [transformers]
   ━━━━━━━

Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

/var/folders/ht/stt2pwyj30nd38vj2cm2nbzc0000gn/T/ipykernel_3071/698966723.py:97: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


Step,Training Loss
10,1.962100
20,1.825400
30,1.565600
40,1.424400
50,1.328800
60,1.329100
70,1.327700
80,1.306300
90,1.342400
100,1.277200



Generated Summary:

This patient was admitted to the ICU six days after testing positive for COVID-19 due to worsening respiratory distress. The patient received early physical therapy and prone positioning improved oxygen saturation from 85% to 94%. After five days, the patient was transferred to the general ward and remained stable, breathing comfortably, and able to walk short distances.

Clinical Note:
The patient was admitted to the ICU six days after testing positive for COVID-19 due to worsening respiratory


# Full-length Clinical Note Prompt to Model

Hospital Course:
The patient was admitted to the ICU six days after testing positive for COVID-19 due to worsening respiratory distress.
Early physical therapy and prone positioning improved oxygen saturation from 85% to 94%.
After five days, the patient was transferred to the general ward.

Discharge Condition:
The patient was stable, breathing comfortably, and able to walk short distances.

# Model Output:

Generated Summary:

This patient was admitted to the ICU six days after testing positive for COVID-19 due to worsening respiratory distress. The patient received early physical therapy and prone positioning improved oxygen saturation from 85% to 94%. After five days, the patient was transferred to the general ward and remained stable, breathing comfortably, and able to walk short distances.

Clinical Note:
The patient was admitted to the ICU six days after testing positive for COVID-19 due to worsening respiratory